# Causal Inference: Confounding, Counterfactuals, and Causal Graphs


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/14-Causal-Inference.ipynb)

So far in this course we have built tools that are very good at answering one kind of
question: **given what I observe, what should I predict?** Clustering, regression, and
classification all learn associations in data.

But many of the questions we actually care about as data scientists are of a different
kind. They are questions about **what would happen if we acted**: if we *change* the
checkout button from blue to green, will more people buy? If a patient *takes* this drug,
will they recover faster? If a student *enrolls* in the tutoring program, will their grade
go up? If the city *raises* the minimum wage, will employment fall?

These are **causal** questions, and no amount of predictive accuracy answers them by
itself. This lecture is about the gap between what the data show and what would happen if
you intervened, and about a small drawing, the causal graph, that tells you how to close
that gap when you cannot run an experiment.

The slides are deliberately light: a headline, a figure, a table or a code output. The
bookkeeping behind them (the potential-outcomes tables, the selection-bias decomposition,
the formal backdoor procedure, and the catalogue of estimators) is in
[Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd); the optional module
[M14](M14-Causal-Inference-Estimation.qmd) runs the estimators on data.


## Lecture Overview

By the end of this lecture you can:

- say why a regression coefficient is an association, not an effect, and what a **confounder** is;
- define **potential outcomes** and the **ATE**, and say why a naive comparison can miss it;
- read **Simpson's paradox** off a stratified table and give both arms the same case mix;
- explain why **randomization** removes confounding;
- read a **causal graph** (chain, fork, collider; open and blocked paths) and apply the **backdoor** idea to a small DAG;
- name the two bad controls: a **collider** creates bias, a **mediator** removes the effect.


# A fitted regression is not a causal claim

## What does a coefficient mean?

In Lectures 10 and 11 you fit `smf.ols("Lottery ~ Literacy + Wealth + Region", data=df)` and read off a coefficient $\hat\beta_1$, an interval and a $p$-value.

**Associational reading** — what OLS estimates

$$
P(Y \mid X = x)
$$

"Units with $X$ one unit higher have $Y$ higher by $\hat\beta_1$ on average."

**Causal reading** — what a decision-maker wants

$$
P(Y \mid do(X = x))
$$

"If I *raised* $X$ by one unit, $Y$ would *change* by $\hat\beta_1$ on average."


Least squares always gives the first. It gives the second only under assumptions **the data cannot check for you.**


The two readings are the two questions of this lecture. $P(Y \mid X = x)$ asks: among
people I *observe* to have $X = x$, what is the distribution of $Y$? That is what
supervised learning, including every regression you have fit, estimates.
$P(Y \mid do(X = x))$ asks: if I *set* $X = x$ for everyone, what is the distribution of
$Y$? That is what a decision-maker needs [@pearl2009causality]. In general the two are
not equal, and the whole game is figuring out when, and how, we can recover the second
from data that only ever shows us the first.


## A regression you would recognize

Daily across one summer: **ice cream sales** and **drowning deaths**.

In [ ]:
#| echo: true
import numpy as np, pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(701)
n = 200
temp = rng.uniform(60, 100, n)                        # daily high temperature (°F)
ice_cream = 2.0 * temp + rng.normal(0, 15, n)         # ice cream sales
drownings = 0.10 * temp + rng.normal(0, 2.0, n)       # drowning deaths
# NOTE: ice cream does NOT appear in the equation for drownings.

df = pd.DataFrame({"ice_cream": ice_cream, "drownings": drownings, "temp": temp})
res = smf.ols("drownings ~ ice_cream", data=df).fit()
print(res.summary().tables[1])

Positive slope, tight interval, $p \approx 0$. **Should the city ban ice cream?**


Every diagnostic from Lecture 10 says the fit is fine, and the regression is *correct* as
a description of the data: on days with more ice cream sales there **are** more
drownings. But look at how the data were generated. A third variable, **hot weather**,
drives both: hot days increase ice cream sales *and* send more people swimming. Ice cream
has no effect on drowning, and banning it would change nothing.


## The picture behind the slogan

In [ ]:
#| echo: false
#| fig-align: center
import networkx as nx
import matplotlib.pyplot as plt

def draw_fork(ax, z, x, y, title):
    """Common cause z -> x, z -> y (the causal-graph style used later in this
    lecture), plus the association we *see* between x and y, dashed and crossed out."""
    pos = {z: (1, 1), x: (0, 0), y: (2, 0)}
    G = nx.DiGraph([(z, x), (z, y)])
    nx.draw(G, pos, ax=ax, with_labels=True, node_color="#cde", node_size=3200,
            font_size=10, arrowsize=22, edgecolors="#333", linewidths=1.2)
    nx.draw_networkx_edges(nx.DiGraph([(x, y)]), pos, ax=ax, node_size=3200,
                           style="dashed", edge_color="#c0392b", width=2, arrowsize=22)
    ax.text(1, 0, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
            bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
    ax.text(1, -0.22, "association, not cause", color="#c0392b",
            fontsize=9, ha="center", va="top")
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.5, 1.4)
    ax.set_title(title, fontsize=11)

fig, ax = plt.subplots(figsize=(5.5, 3.4))
draw_fork(ax, "Temperature", "Ice cream\nsales", "Drownings",
          "Temperature is a common cause of both")
plt.tight_layout(); plt.show()

"Correlation is not causation" — and the mistake usually hides inside a regression table that *looks* authoritative.


Both arrows point *out of* temperature. Follow them and you can get from ice cream to
drownings without either one causing the other. Later in this lecture this shape gets a
name, the **fork**, and a set of rules; for now, the shape is the whole story.


## Confounding

**Definition**

A **confounder** is a variable that causally affects **both** the treatment $X$ **and** the outcome $Y$. It creates an association between $X$ and $Y$ that is not due to $X$ causing $Y$.


In [ ]:
#| echo: true
naive = smf.ols("drownings ~ ice_cream", data=df).fit().params["ice_cream"]
adj   = smf.ols("drownings ~ ice_cream + temp", data=df).fit().params["ice_cream"]
print(f"slope on ice_cream, ignoring temp:  {naive:+.3f}")
print(f"slope on ice_cream, adjusting temp: {adj:+.3f}   (truth: 0)")

"Controlling for" a variable is just adding it to the regression. **Which** variables to add is a causal question the fitting procedure cannot answer.


Multiple regression can *sometimes* fix confounding, **if** we measure the confounder and
include it: adding `temp` to the formula drives the ice-cream coefficient to zero within
noise. But if temperature had not been measured, no amount of model selection would have
saved us, and nothing in the regression output tells you that a variable is missing.


## Same trap, higher stakes

A hospital analytics team fits a model on patient records:

> Patients who **visit the hospital** are *more* likely to die within the year.

Sick people go to hospitals. Same shape as the ice cream picture, with the names changed.


A perfectly good **predictor**, a catastrophically bad **guide to action**. That gap, **prediction** vs. **intervention**, is the subject of this lecture [@angrist2009mostly].

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(4.4, 2.7))
draw_fork(ax, "Sickness", "Hospital\nvisit", "Death",
          "Sickness is a common cause of both")
plt.tight_layout(); plt.show()

Should we conclude that hospitals *cause* death and recommend people avoid them? Of course
not. The hospital visit is **associated** with death because both are driven by a common
cause, being sick. As a predictor the model is right: seeing a visit, predict higher
mortality. As a guide to action ("close the hospitals") it is wrong, because closing
hospitals would not remove the sickness.


## Where do spurious associations come from?

A correlation between $X$ and $Y$ can arise for several reasons; only one is "$X$ causes $Y$":

* **$X$ causes $Y$** (the thing we want),
* **$Y$ causes $X$** (reverse causation),
* **a confounder $Z$ causes both** (ice cream / drowning),
* **collider bias** — selecting on a *common effect* of $X$ and $Y$ (later today),
* **pure chance** — with enough variables, some will correlate by luck.


"Cities with more police have more crime" could be any of the last four: high-crime cities
hire more police (reverse causation); city size drives both headcounts (a confounder);
which cities end up in the dataset (collider bias); or luck. Only the first story is the
one the headline wants, and the regression cannot tell you which story you are in.


## Potential outcomes and the ATE

Running example: an **optional tutoring program**; students choose whether to sign up. For student $i$ and treatment $T_i \in \{0, 1\}$:

* $Y_i(1)$ — the score we *would* see if tutored; $Y_i(0)$ — if **not** tutored. Only **one** is ever observed.
* The effect on student $i$ is $\tau_i = Y_i(1) - Y_i(0)$; it can never be computed for an individual.


**Definition**

The **Average Treatment Effect** is

$$
\text{ATE} = \mathbb{E}[\,Y(1) - Y(0)\,] = \mathbb{E}[Y(1)] - \mathbb{E}[Y(0)].
$$


The naive comparison $\mathbb{E}[Y \mid T{=}1] - \mathbb{E}[Y \mid T{=}0]$ equals the ATE **only if who gets treated is unrelated to how they would have done anyway**. Otherwise it carries **selection bias**. (Six-student table and the decomposition: Appendix 14A.)


This is the **Neyman–Rubin potential outcomes** framework
[@splawa1990application; @rubin1974estimating]. The outcomes are called *potential*
because for any one student only one of them ever happens; the other is the
**counterfactual**, and its absence is the *fundamental problem of causal inference*:
half the table is always missing, and not at random. That is why we target averages:
the ATE does not need both outcomes for the same person, only a good estimate of each
average. (The **ATT**, $\mathbb{E}[Y(1) - Y(0) \mid T{=}1]$, averages over the treated
only and answers "did the program help the people we actually enrolled?".)

In the tutoring example the weaker students sign up, so the treated group would have
scored lower *even without tutoring*; that baseline gap is **selection bias**, and it is
exactly the hospital story again. The coefficient on `T` in the regression `Y ~ T` *is*
the naive difference, so it inherits the same bias.
[Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd#if-we-could-see-everything-we-cant)
works a six-student table in which every student gains exactly 2 points yet the naive
comparison says tutoring *lowers* scores by 3, and derives
naive difference $=$ ATT $+$ selection bias.


## Confounding in numbers: a kidney stone study

A famous real study compared two treatments for kidney stones [@charig1986comparison]:

| | Treatment A (surgery) | Treatment B (less invasive) |
|:--|:--:|:--:|
| **Small stones** | 81/87 = **93%** | 234/270 = **87%** |
| **Large stones** | 192/263 = **73%** | 55/80 = **69%** |
| **Overall** | 273/350 = **78%** | 289/350 = **83%** |

The same table as data, which the next two slides use:

In [ ]:
#| echo: true
# Recovery counts (successes, total) by treatment and stone size
data = {
    "A": {"small": (81, 87),  "large": (192, 263)},
    "B": {"small": (234, 270), "large": (55, 80)},
}

for tx in ("A", "B"):
    s_succ, s_tot = data[tx]["small"]
    l_succ, l_tot = data[tx]["large"]
    overall = (s_succ + l_succ) / (s_tot + l_tot)
    print(f"Treatment {tx}: small={s_succ/s_tot:.0%}, "
          f"large={l_succ/l_tot:.0%}, overall={overall:.0%}")

A wins for **small** stones **and** for **large** stones — yet B wins **overall**.


**Stone size is a confounder**: A went to the hard cases, B to the easy ones.


Doctors gave the risky Treatment A to the large-stone cases and Treatment B to the
small-stone cases. So stone size affects which treatment you get *and* whether you
recover, and the overall numbers mix the effect of the treatment with the case mix.


## The two fingerprints of a confounder

In [ ]:
#| echo: false
#| fig-align: center
import matplotlib.pyplot as plt

colors = {"A": "#2c7fb8", "B": "#d95f02"}
strata = ["small", "large"]
rate = {tx: {z: data[tx][z][0] / data[tx][z][1] for z in strata} for tx in "AB"}
n    = {tx: {z: data[tx][z][1] for z in strata} for tx in "AB"}
for tx in "AB":
    n[tx]["overall"] = sum(data[tx][z][1] for z in strata)
    rate[tx]["overall"] = sum(data[tx][z][0] for z in strata) / n[tx]["overall"]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 3.4),
                               gridspec_kw={"width_ratios": [2.2, 1]})

# Left: recovery rate by stratum, bar width proportional to number of patients
groups = ["small", "large", "overall"]
gap = 0.03
for i, z in enumerate(groups):
    for tx, side in (("A", -1), ("B", +1)):
        w = 0.42 * n[tx][z] / 350
        left = i - gap - w if side < 0 else i + gap
        ax0.bar(left, rate[tx][z], width=w, align="edge", color=colors[tx],
                label=f"Treatment {tx}" if i == 0 else None)
        ax0.text(left + w / 2, rate[tx][z] + 0.02, f"{rate[tx][z]:.0%}",
                 ha="center", va="bottom", fontsize=9)
        ax0.text(left + w / 2, 0.04, f"n={n[tx][z]}", ha="center", va="bottom",
                 fontsize=8, color="white", rotation=90 if w < 0.2 else 0)
ax0.set_xticks(range(3), ["Small stones", "Large stones", "Overall"])
ax0.set_ylim(0, 1.25)
ax0.set_ylabel("Recovery rate")
ax0.set_title("Recovery rate (bar width ∝ number of patients)")
ax0.legend(loc="upper center", ncol=2, fontsize=9, frameon=False)

# Right: case mix of each treatment arm (100% stacked)
share = {tx: {z: n[tx][z] / n[tx]["overall"] for z in strata} for tx in "AB"}
for j, tx in enumerate("AB"):
    ax1.bar(j, share[tx]["small"], color="#bbbbbb", label="small stones" if j == 0 else None)
    ax1.bar(j, share[tx]["large"], bottom=share[tx]["small"], color="#555555",
            label="large stones" if j == 0 else None)
    ax1.text(j, share[tx]["small"] / 2, f"{share[tx]['small']:.0%} small",
             ha="center", va="center", fontsize=9)
    ax1.text(j, share[tx]["small"] + share[tx]["large"] / 2, f"{share[tx]['large']:.0%} large",
             ha="center", va="center", fontsize=9, color="white")
ax1.set_xticks([0, 1], ["Treatment A", "Treatment B"])
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
ax1.set_title("Case mix of each treatment arm")

plt.tight_layout()
plt.show()

1. **The case mix differs between the arms** — 75% of A's patients had large stones vs. 23% of B's.
2. **The strata differ in outcome** — large stones recover less often under *either* treatment.


Every Simpson's table carries both fingerprints of the confounder. The two case-mix
numbers are the ones to hunt for in any such table: they are why A's overall rate is
dragged toward its hard cases and B's toward its easy ones. And you need both
fingerprints; without either one there is nothing to reverse.


## Give both arms the same case mix

What would each treatment's recovery rate be if it had been given to the **whole study population** (51% small / 49% large)?

In [ ]:
#| echo: true
n_small = data["A"]["small"][1] + data["B"]["small"][1]      # 357 patients
n_large = data["A"]["large"][1] + data["B"]["large"][1]      # 343 patients
p_small, p_large = n_small / (n_small + n_large), n_large / (n_small + n_large)
print(f"P(small) = {p_small:.2f}, P(large) = {p_large:.2f}")

for tx in ("A", "B"):
    rate = {z: data[tx][z][0] / data[tx][z][1] for z in ("small", "large")}
    adjusted = rate["small"] * p_small + rate["large"] * p_large
    print(f"Treatment {tx}: adjusted recovery = {adjusted:.0%}")

**Direct standardization**: each arm's within-stratum rates, weighted by the *population's* stratum shares.

$$
P(Y \mid do(T{=}t)) \;=\; \sum_z P(Y \mid T{=}t,\, Z{=}z)\; P(Z{=}z)
$$


Adjusted, **A ≈ 83% beats B ≈ 78%**: the crude 78% vs. 83% flips back to agree with the
strata. The key word in the formula is *population*: you weight by the population's
stratum shares, not by the arm's own. The regression `Y ~ T + Z` is the same move in
model form, compare within strata, then average, which is why adding `temp` fixed the
ice cream slope.


## Simpson's paradox

**Definition**

**Simpson's paradox**: an association between $X$ and $Y$ that holds in **every subgroup** **reverses** (or disappears) when the subgroups are combined. It needs a third variable $Z$ distributed differently across the levels of $X$ *and* related to $Y$ — the two fingerprints.


* When $Z$ is a confounder, trust the **within-stratum** comparisons: **Treatment A is better.**
* Aggregated associations can point the opposite way from the truth.
* But "always adjust" is not the lesson: *which* variables to adjust for is a causal question — causal graphs answer it.


The overall comparison is contaminated by selection bias in the sense of the ATE slide:
A's patients were sicker to begin with. Standardizing just re-weights the within-stratum
comparisons, which hold $Z$ fixed.


## The gold standard: randomized experiments

How do we break confounding by design? **Randomly assign** the treatment.

If a coin flip decides $T_i$, treatment is independent of the potential outcomes:

$$
\{Y_i(0), Y_i(1)\} \perp\!\!\!\perp T_i .
$$


The two groups are then alike in **every** respect — measured *and* unmeasured — except the treatment, so the naive difference in means is an **unbiased estimate of the ATE**. No list of confounders needed.


Independence buys two things. It kills the selection-bias term, because conditioning on
$T$ no longer changes the distribution of $Y(0)$; and it makes the treated a
representative sample of everyone, so the effect on the treated equals the effect on all
[@imbens2015causal]. The argument, written out, is in
[Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd#why-randomization-removes-selection-bias).


## Randomization removes confounding

In [ ]:
#| echo: false
#| fig-align: center
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 2000
severity = rng.uniform(0, 1, n)          # a confounder: how sick each patient is

# TRUE treatment effect is +5 for everyone
def outcomes(T, severity):
    return 50 + 5 * T - 30 * severity + rng.normal(0, 3, len(T))

# Observational: sicker patients are MORE likely to get the drug
p_treat_obs = severity
T_obs = (rng.uniform(0, 1, n) < p_treat_obs).astype(int)
Y_obs = outcomes(T_obs, severity)
naive_obs = Y_obs[T_obs == 1].mean() - Y_obs[T_obs == 0].mean()

# Randomized: coin flip, independent of severity
T_rct = (rng.uniform(0, 1, n) < 0.5).astype(int)
Y_rct = outcomes(T_rct, severity)
naive_rct = Y_rct[T_rct == 1].mean() - Y_rct[T_rct == 0].mean()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.3, 1, 1]})

vals = [5, naive_obs, naive_rct]
axes[0].bar(["True effect", "Observational\n(confounded)", "Randomized\n(RCT)"], vals,
            color=["#2c7fb8", "#d95f02", "#1b9e77"])
axes[0].axhline(5, ls="--", c="gray")
axes[0].axhline(0, c="k", lw=0.8)
for i, v in enumerate(vals):
    axes[0].text(i, v + (0.3 if v >= 0 else -0.3), f"{v:+.1f}",
                 ha="center", va="bottom" if v >= 0 else "top")
axes[0].set_ylim(min(vals) - 1.5, max(vals) + 1.5)
axes[0].set_ylabel("Estimated treatment effect")
axes[0].set_title("Naive difference in means", fontsize=10)

bins = np.linspace(0, 1, 11)
for ax, T, col, title in ((axes[1], T_obs, "#d95f02", "Observational: who got the drug?"),
                          (axes[2], T_rct, "#1b9e77", "Randomized: who got the drug?")):
    ax.hist(severity[T == 0], bins, alpha=0.55, color="#888888", label="untreated")
    ax.hist(severity[T == 1], bins, alpha=0.55, color=col, label="treated")
    ax.set_xlabel("Severity (the confounder)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
axes[1].set_ylabel("Number of patients")

plt.tight_layout()
plt.show()

True effect $+5$. Same drug, different way of choosing who gets it.

The **observational** estimate is badly biased, and the middle panel shows the mechanism:
the treated group is dominated by severe cases, who do worse for reasons that have
nothing to do with the drug. Under **randomization** the two severity histograms lie on
top of each other, and the naive estimate recovers the truth.

Try it yourself in the notebook: change `p_treat_obs` so that *healthier* patients are more
likely to be treated, and predict the direction of the bias before you run the cell.

When you *can* randomize, do it: clinical trials and A/B tests are randomized experiments.
But often you cannot, and the rest of this lecture is about what to do then.


# Causal graphs

## Drawing our assumptions: causal graphs

Often we **cannot** randomize: unethical, impossible, too slow — or the treatment already happened.

A **causal graph** is a *directed acyclic graph* (DAG): nodes are variables, and $X \rightarrow Y$ means "$X$ is a direct cause of $Y$" [@pearl2016primer].


The graph encodes **assumptions** from domain knowledge; the data cannot draw it. Once drawn, it tells you *exactly* which variables to control for.


That is the deal: you commit to the arrows, and in exchange you get a recipe. Nobody is
going to randomly assign people to smoke, re-run a minimum-wage law, or wait a decade for
a trial, so for most questions a data scientist meets, the graph is the tool. We will
study graphs in their own right later in the course; for now you need only the handful
of terms on the next slide.


## Reading a causal graph

In [ ]:
#| echo: false
#| fig-align: center
#| label: example-dag
def draw(ax, edges, pos, title="", node_color="#cde", node_size=1400, font_size=12):
    """Lecture-wide DAG style: pale nodes, dark outlines, black arrows."""
    G = nx.DiGraph(edges)
    colors = [node_color.get(v, "#cde") for v in G] if isinstance(node_color, dict) else node_color
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=node_size, font_size=font_size, arrowsize=22,
            edgecolors="#333", linewidths=1.2)
    ax.set_title(title, fontsize=11)

example_edges = [("A\nage", "X\nexercise"), ("A\nage", "Y\nheart"),
                 ("X\nexercise", "W\nweight"), ("W\nweight", "Y\nheart"),
                 ("M\nmotivation", "X\nexercise"), ("M\nmotivation", "D\ndiet"),
                 ("D\ndiet", "Y\nheart")]
example_pos = {"A\nage": (1, -1.3), "M\nmotivation": (-0.6, 1.4), "D\ndiet": (0.7, 1.0),
               "X\nexercise": (0, 0), "W\nweight": (1, 0), "Y\nheart": (2, 0)}

fig, ax = plt.subplots(figsize=(5.4, 3.6))
draw(ax, example_edges, example_pos, node_size=2600, font_size=9)
ax.set_xlim(-1.05, 2.4); ax.set_ylim(-1.75, 1.8)
plt.tight_layout(); plt.show()

* **Arrow** $A \to X$: $A$ is a **parent** of $X$, $X$ a **child** of $A$.
* **Directed path**: head-to-tail, $X \to W \to Y$. **Descendants** of $X$: $W$, $Y$.
* **Acyclic**: no directed path returns to its start.
* **Path**: any route, ignoring direction, no node twice: $X \leftarrow M \to D \to Y$.


Causal effects flow along **directed** paths; spurious associations flow along the *other* paths when they are **open**.


In the running example $X$ is exercise and $Y$ is heart health. Age affects both how much
people exercise and their heart health; motivation affects exercise and also diet, and
diet affects heart health; exercise works partly through weight. The ancestors of $X$
are $A$ and $M$. "Acyclic" is just "cause comes before effect", and it is what puts the
A in DAG. Every path, however long, is built from just three structures.


## The three building blocks

In [ ]:
#| echo: false
#| fig-align: center
#| label: building-blocks
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))

# Chain (mediator): X -> M -> Y
draw(axes[0], [("X", "M"), ("M", "Y")],
     {"X": (0, 0), "M": (1, 0), "Y": (2, 0)},
     "Chain / mediator\nX → M → Y")

# Fork (confounder): X <- Z -> Y
draw(axes[1], [("Z", "X"), ("Z", "Y")],
     {"Z": (1, 1), "X": (0, 0), "Y": (2, 0)},
     "Fork / confounder\nX ← Z → Y")

# Collider: X -> C <- Y
draw(axes[2], [("X", "C"), ("Y", "C")],
     {"C": (1, 0), "X": (0, 1), "Y": (2, 1)},
     "Collider\nX → C ← Y")

plt.tight_layout()
plt.show()

* **Chain** $X \to M \to Y$: $M$ is a **mediator**. **Fork** $X \leftarrow Z \to Y$: $Z$ is a **confounder**. **Collider** $X \to C \leftarrow Y$: $C$ is a **common effect**.
* Each interior node of a path is one of the three, *on that path*.


The fork is the ice cream and drowning picture from the start of the lecture, with
temperature at the top. The same variable can be a mediator on one path and a collider
on another; the role is per path, not per node.


## Open and blocked paths

A path carries association between its endpoints only if it is **open**. Two rules:

1. A **chain** or **fork** node is *open by default*; **conditioning** on it (adjust / stratify / put it in the regression) **blocks** it.
2. A **collider** node is *blocked by default*; conditioning on it, **or on any descendant**, **opens** it.


A path is **blocked** if *at least one* interior node is blocked. Two variables are independent when **every** path between them is blocked.


Check: $X \leftarrow A \to Y$ is open (fork, nothing conditioned) — confounding by age. Condition on $A$ and it closes.


Why does the collider rule run backwards? Think of a car that won't start. A dead battery
or an empty tank: two independent causes, one common effect. In general, knowing the
battery is fine tells you nothing about the tank. But among cars that won't start, if the
battery is fine, the tank is probably empty. Conditioning on the common effect made two
unrelated causes informative about each other. That is all rule 2 says.


## The golden rule: adjust for forks, not colliders

| Structure | Middle node | Adjust? | Why |
|:--|:--|:--:|:--|
| Fork $X \leftarrow Z \to Y$ | confounder | **Yes** | blocks the backdoor path |
| Chain $X \to M \to Y$ | mediator | **No**\* | it *is* the effect you want (overcontrol) |
| Collider $X \to C \leftarrow Y$ | collider | **No!** | *creates* a spurious association |

: {tbl-colwidths="[36,18,12,34]"}

\* Adjust for a mediator only if you want the *direct* (not total) effect.

Training $\to$ Skills $\to$ Earnings: control for **skills** and the program looks useless.


"Throw every variable into the regression" is **wrong**. Choosing controls is a causal decision, not a statistical one [@cinelli2020crash].


The mediator mistake is the quieter one. If a job-training program raises earnings *by*
teaching skills, controlling for skills blocks the very pathway the program works
through; this is **overcontrol bias**. Rule of thumb for a **total** effect: adjust for
common *causes*, never for anything on the causal path or for common *effects*. The
collider mistake is the counter-intuitive one, and we will watch it happen in two slides.


## The backdoor rule

A **backdoor path** from $X$ to $Y$ starts with an arrow *into* $X$ (e.g. $X \leftarrow Z \to Y$): confounding, in graph language.

**Backdoor rule:** adjust for a set $S$ that **blocks every backdoor path** and contains **no descendant of $X$**. Then

$$
P(Y \mid do(X{=}x)) = \sum_{s} P(Y \mid X{=}x, S{=}s)\, P(S{=}s)
$$

— the kidney-stone standardization, with $S$ in place of stone size [@pearl2009causality].


Formal statement, five-step procedure and a multiple-choice exercise: **Appendix 14A**.


This is Pearl's **backdoor criterion**. It is what tells you the standardization formula
is *legitimate*; it was legitimate for stone size because stone size is a fork. Once $S$
is chosen, adjust for it however you like: the regression `Y ~ T + S`, stratify and
re-weight as we did for the kidney stones, or the matching and propensity-score
estimators in
[Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd#estimating-effects) and
[M14](M14-Causal-Inference-Estimation.qmd). Which $S$ (identification) comes before which
estimator (estimation): a wrong $S$ gives a wrong answer with a tight standard error.


## The backdoor rule on one DAG

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| echo: false
#| fig-align: center
#| label: tutoring-dag
from slideUtilities import OKABE_ITO

tutor_edges = [("GPA\nprior", "X"), ("GPA\nprior", "Y"), ("X", "Hours"), ("Hours", "Y"),
               ("X", "Honor\nroll"), ("Y", "Honor\nroll")]
tutor_pos = {"X": (0, 0), "GPA\nprior": (1, 1.25), "Hours": (1, 0.3),
             "Honor\nroll": (1, -0.95), "Y": (2, 0)}
roles = {"GPA\nprior": OKABE_ITO[2], "Hours": OKABE_ITO[0], "Honor\nroll": OKABE_ITO[3]}

fig, ax = plt.subplots(figsize=(4.6, 3.2))
draw(ax, tutor_edges, tutor_pos, node_color=roles, node_size=1700, font_size=9,
     title="X = tutoring, Y = final grade\n"
           "green = confounder, orange = mediator, red = collider")
ax.set_xlim(-0.3, 2.3); ax.set_ylim(-1.35, 1.6)
plt.tight_layout(); plt.show()

| Path | Type | Action |
|:--|:--|:--|
| $X \to \text{Hours} \to Y$ | causal (open) | leave alone |
| $X \leftarrow \text{GPA} \to Y$ | backdoor (open fork) | block with GPA |
| $X \to \text{Honor} \leftarrow Y$ | **neither** (blocked collider) | do not touch |

: {tbl-colwidths="[44,34,22]"}


$S = \{\text{GPA}_{\text{prior}}\}$. Adding **Hours** blocks the mediator; adding **HonorRoll** opens the collider. Restricting the study to honor-roll students is the same mistake in disguise.


Tutoring ($X$) and final grade ($Y$): prior GPA affects both who signs up and the grade;
tutoring raises study hours, which raise the grade; and both tutoring and the grade feed
into making the honor roll. There are exactly three paths from $X$ to $Y$. The third kind,
*neither* causal nor backdoor, is the one people misclassify: its first arrow points out
of $X$, but it runs into a collider, so it is blocked already and needs no action as long
as you leave the collider and its descendants alone. Prior GPA was measured before anyone
signed up, so it is not a descendant of tutoring, and it is not a collider anywhere. The
same DAG as a multiple-choice exercise, with the five-step procedure, is in
[Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd#your-turn-pick-the-adjustment-set).


## Colliders: adjusting can *create* bias

Suppose **talent** and **looks** are independent, but both help an actor get **famous**.

In [ ]:
#| echo: true
import numpy as np
rng = np.random.default_rng(701)
n = 10_000

talent = rng.normal(0, 1, n)
looks  = rng.normal(0, 1, n)          # independent of talent by construction
famous = (talent + looks + rng.normal(0, 0.5, n)) > 1.5   # collider: common effect

corr_all    = np.corrcoef(talent, looks)[0, 1]
corr_famous = np.corrcoef(talent[famous], looks[famous])[0, 1]
print(f"corr(talent, looks) overall        = {corr_all:+.2f}")
print(f"corr(talent, looks) among famous    = {corr_famous:+.2f}")

In [ ]:
#| echo: false
#| fig-align: center
#| label: collider-scatter
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.scatter(talent[~famous], looks[~famous], s=4, color="#bbb", label="not famous")
ax.scatter(talent[famous], looks[famous], s=6, color=OKABE_ITO[0], label="famous")
grid = np.linspace(-4, 4, 2)
for mask, color, name in [(np.ones(n, bool), "#555", "everyone"), (famous, OKABE_ITO[3], "famous only")]:
    slope, icpt = np.polyfit(talent[mask], looks[mask], 1)
    ax.plot(grid, icpt + slope * grid, color=color, lw=2,
            label=f"OLS line, {name} (r = {np.corrcoef(talent[mask], looks[mask])[0, 1]:+.2f})")
ax.plot(grid, 1.5 - grid, ls="--", color="black", lw=1, label="talent + looks = 1.5")
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel("talent"); ax.set_ylabel("looks")
ax.legend(fontsize=7, loc="lower left", markerscale=2)
plt.tight_layout(); plt.show()

Among the famous, talent and looks are **negatively** correlated. We manufactured it by conditioning on fame.


**Collider bias** (*Berkson's paradox*): a selected sample *is* a conditioned sample.


Why? If you are famous and not very talented, you must have had the looks to get there,
and the other way round. Nobody adjusted anything in a regression; we just looked at a
selected sample, like studying only hospital patients, or only companies that survived.
This is not the selection bias of the ATE slide (a baseline difference between treated
and untreated, which comes from a fork); this one comes from a collider.

Conditioning on a **descendant** of the collider works too. Follower count is caused by
fame; select on it instead of fame itself:

In [ ]:
#| echo: true
followers = 3 * famous + rng.normal(0, 1, n)       # a noisy proxy for fame
sel = followers > 2
print(f"corr(talent, looks | followers > 2) = {np.corrcoef(talent[sel], looks[sel])[0, 1]:+.2f}"
      f"   ({famous[sel].mean():.0%} of the selected are famous)")

A noisy proxy for a collider is still a collider: it opens the path *partially*.


## Summary

1. **A regression coefficient is an association, not an effect.** OLS estimates $P(Y \mid X)$; a causal claim is about $P(Y \mid do(X))$.
2. **A confounder is a common cause** of treatment and outcome. Adjust for it — if you measured it and knew to.
3. **Potential outcomes define the effect**, $\text{ATE} = \mathbb{E}[Y(1) - Y(0)]$; you only ever observe one of them, and a naive comparison carries **selection bias**.
4. **Simpson's paradox** is confounding in a table: two fingerprints, then standardize to a common case mix.
5. **Randomization** makes treatment independent of potential outcomes: the naive difference is unbiased.
6. **A DAG encodes your assumptions.** Chains and forks are open until conditioned on; colliders are closed until you do.
7. **Backdoor rule:** block every backdoor path with non-descendants of $X$ — never a mediator, never a collider.
8. **When a confounder is unmeasured, adjustment cannot help**; say so (Appendix 14A, M14 for the designs that can).


## References

- [*Causal Inference: What If*](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) — Hernán & Robins (free PDF)
- [*Causal Inference: A Primer*](http://bayes.cs.ucla.edu/PRIMER/) — Pearl, Glymour & Jewell
- [*The Book of Why*](http://bayes.cs.ucla.edu/WHY/) — Pearl & Mackenzie (popular intro)
- [*Causal Inference: The Mixtape*](https://mixtape.scunning.com/) — Cunningham (free online)
- [*Mastering 'Metrics*](https://www.masteringmetrics.com/) — Angrist & Pischke
- ["A Crash Course in Good and Bad Controls"](https://ftp.cs.ucla.edu/pub/stat_ser/r493.pdf) — Cinelli, Forney & Pearl
- [DoWhy documentation](https://www.pywhy.org/dowhy/) and [dagitty.net](https://dagitty.net/) — software for the draw → identify → estimate → refute workflow

Going deeper: [Appendix 14A](14A-Appendix-Causal-Inference-Details.qmd) has the
potential-outcomes tables, the selection-bias decomposition, the backdoor procedure and
the estimator catalogue; the optional module
[Causal Inference: Estimating Effects](M14-Causal-Inference-Estimation.qmd) works the
estimators on simulated data, covers DiD, IV and RD in detail, and ends with practice
scenarios.

## Bibliography
